# MOUSE CORE · Experiment 2 v3

## Lexikální rozmanitost, kontextová proměnlivost a dvě lidové reference

Nová, samostatně sestavená verze experimentu. Zachovává celý výzkumný rozsah: devět autorských pohádek a dvě samostatné lidové reference: „O kohoutkovi a slepičce“ a uživatelskou verzi „O veliké řepě“.

**Základní jednotkou analýzy zůstává celý příběh.** Věty slouží pouze jako lokální kontexty opakovaných slovních forem, nikoli jako náhrada celého příběhu.

Zachováváme převzetí celotextových vektorů z Pilot v2 bez přepočítání, lexikální profily, opakované formy, hapax a frekvenční statistiky, kontexty výskytů, nové větné embeddingy BGE-M3, párové podobnosti, kontextovou proměnlivost, srovnání skupin, vizualizace, exporty a všech dvanáct úhlů Semantic Gap jako interpretační rámec.

Metodické omezení: odlišnost větných embeddingů není přímým důkazem změny významu konkrétního slova. TTR měří rozmanitost slovních forem, nikoli významovou hloubku.

## 0 · Připojení prostředí a kontrola

Spouštěj shora dolů. Pokud Colab zůstane viset už na připojování prostředí, problém je v běhovém prostředí Colabu, ne v analytických výpočtech.

In [ ]:
import os, sys, time, json, re, io, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from urllib.request import Request, urlopen
print("Python:", sys.version)
try:
 import torch
 print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
 if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except Exception as e: print("PyTorch zatím není dostupný:", repr(e))

## 1 · Instalace knihoven

Nepoužíváme plošné aktualizace balíčků. Instalujeme pouze knihovnu pro BGE-M3 a zachováváme předinstalované prostředí Colabu.

In [ ]:
%pip -q install FlagEmbedding pandas matplotlib
print("Instalace dokončena.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from google.colab import files
from FlagEmbedding import BGEM3FlagModel
print("Importy v pořádku.")
print("Zařízení:", "GPU" if torch.cuda.is_available() else "CPU")

## 2 · Nahrání výsledků Pilot v2

Nahraj přesně MOUSE_CORE_v2_vysledky.zip. ZIP nerozbaluj ručně. Převezmeme devět původních celotextových vektorů a vektor lidové reference. Tyto celotextové vektory se znovu nepočítají.

In [ ]:
print("Vyber soubor MOUSE_CORE_v2_vysledky.zip")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
assert len(zip_names) == 1, "Nahraj právě jeden ZIP s výsledky Pilot v2."
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as zf:
 names = zf.namelist()
 vf = next((n for n in names if n.endswith("mouse_core_bge_m3_vectors_v2.json")), None)
 rf = next((n for n in names if n.endswith("mouse_core_bge_m3_reference_comparison_v2.json")), None)
 af = next((n for n in names if n.endswith("mouse_core_bge_m3_analysis_v2.json")), None)
 mf = next((n for n in names if n.endswith("mouse_core_bge_m3_manifest_v2.json")), None)
 assert vf and rf, "ZIP neobsahuje očekávané soubory Pilot v2: " + str(names)
 baseline_vectors = json.loads(zf.read(vf))
 baseline_reference = json.loads(zf.read(rf))
 baseline_analysis = json.loads(zf.read(af)) if af else None
 baseline_manifest = json.loads(zf.read(mf)) if mf else None
assert len(baseline_vectors) == 9
assert len(baseline_reference["vector"]) == 1024
assert all(len(x["vector"]) == 1024 for x in baseline_vectors)
print("ZIP ověřen: 9 autorských vektorů, 1 vektor reference, dimenze 1024.")

## 3 · Načtení deseti celých textů

Texty načítáme z repozitáře s opakováním při síťové chybě. Z autorských textů odstraňujeme pouze celé technické řádky začínající Karta: nebo Text:. Literární obsah zůstává zachován. Lidová reference se načítá samostatně.

In [ ]:
REPO = "https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main"
OUT = Path("/content/MOUSE_CORE_Experiment2_v3"); OUT.mkdir(parents=True, exist_ok=True)
CARDS = {
 "mys":("Myš","doma/primarni_data/kriva_pohadka/mys.txt"),
 "ryba":("Ryba","doma/primarni_data/kriva_pohadka/ryba.txt"),
 "kocour":("Kocour","doma/primarni_data/kriva_pohadka/kocour.txt"),
 "lemur":("Lemur","doma/primarni_data/kriva_pohadka/lemur.txt"),
 "slon":("Slon","doma/primarni_data/kriva_pohadka/slon.txt"),
 "pav":("Páv","doma/primarni_data/kriva_pohadka/pav.txt"),
 "cap":("Čáp","doma/primarni_data/kriva_pohadka/cap.txt"),
 "kozel":("Kozel","doma/primarni_data/kriva_pohadka/kozel.txt"),
 "zajic":("Zajíc","doma/primarni_data/kriva_pohadka/zajic.txt")}
REFERENCE=("kohoutek_slepicka","O kohoutkovi a slepičce","doma/primarni_data/reference/lidova_pohadka_kohoutek_a_slepicka.txt")
def fetch_text(path, attempts=4):
 last=None
 for attempt in range(1,attempts+1):
  try:
   req=Request(f"{REPO}/{path}",headers={"User-Agent":"MOUSE-CORE-Experiment2"})
   with urlopen(req,timeout=35) as response: return response.read().decode("utf-8-sig")
  except Exception as exc:
   last=exc; print(f"Opakování {attempt}/{attempts}: {path} | {exc}"); time.sleep(attempt*2)
 raise RuntimeError(f"Text nelze načíst: {path}") from last
docs=[]
for cid,(name,path) in CARDS.items():
 raw=fetch_text(path)
 clean=re.sub(r"(?im)^[ \t]*(?:Karta|Text):[^\r\n]*(?:\r?\n|$)","",raw).strip()
 assert clean, f"Prázdný text: {cid}"
 docs.append({"id":cid,"name":name,"group":"authored","source":path,"text":clean})
 print("Načteno:",name,"|",len(clean),"znaků")
rid,rname,rpath=REFERENCE
reference_text=fetch_text(rpath).strip()
assert reference_text
docs.append({"id":rid,"name":rname,"group":"reference","source":rpath,"text":reference_text})
# Druhá reference: přesná verze pohádky dodaná autorkou v této konverzaci.
TURNIP_TEXT = "Byli jeden dědeček a babička a měli malou vnučku. Hospodařili na svém malém políčku a chovali několik domácích zvířátek. Každý den vnučka Evička pomáhala svým prarodičům krmit husy, slepice a dobroty také nosila pejskovi, který měl před domkem svou misku. Pejsek nebyl lakomý a pokaždé schoval něco na zub pro kočičku. Ta pak odsypala ze svého pár drobtů k myší komůrce na půdě. Nakrmená myška si pak s kočičkou hrála na honěnou třeba celou noc. A tak tu spolu hezky žili.\n\nZatímco Evička krmila zvířátka, babička pracovala v kuchyni, děda, ten měl na starost pole. Brzy ráno se svou prací začínal a pozdě večer s ní končil. Sázel, zaléval, plel, okopával a o co víc pracoval, o to víc se rostlinkám dařilo. Vypěstoval řepu takovou, že se na ni chodili dívat ze sousední vsi. Byla obrovská, zabírala půlku pole a stále rostla a zvětšovala se. Nedalo se nic dělat, děda ji musel vykopat. Přece jen to jeho pole zas tak veliké nebylo. Motyka se mu však hned zlomila. Co teď? Jak tu řepu dostat ven?\n\nZavolal na pomoc babičku. Ta uchopila dědu v pase, ten chytil ze všech sil silné listy a společnými silami táhli, táhli, ale řepu nevytáhli.\nBabička zavolala Evičku. \"Ráda pomohu, však vy se o mne také staráte,\" přiběhla ochotně Evička. Ač se snažili ze všech sil, řepa nepovolila.\nVnučka běžela pro pejska. Pes chytil Evičku, Evička babičku, babička dědečka, dědeček řepu - táhli - táhli, táhli, ale řepu stejně nevytáhli.\n\"Haf, haf, sami nic nezmůžeme. Kočičko, pojď nám na pomoc!\" přivolal pejsek kočičku. Ta se postavila do řady, chytila pejska, ten vnučku, ta babičku, ta dědečka, ten řepu - táhli, táhli - už se zdálo, že řepa trochu povoluje - ale nakonec ji nevytáhli.\n\n\"Budeme to muset vzdát. Na takovou velikou řepu nemáme dost sil,\" vzdychal zklamaně děda.\n\"A co já?,\" ozval se dole u země pisklavý hlásek. To přišla kočičce na pomoc malá myška.\n\"Ty myško? Jsi taková maličká. Jak bys mohla pomoci?\" Nevěřila babička.\nDěda se jen smál a taky nevěřil.\nAle myška už stála v řadě. Chytila se pevně kočičky, kočka pejska, pejsek vnučky, vnučka babičky, babička dědečka, dědeček řepy - táhli, tááhli, táááhli - a najednou rup! - řepa byla venku.\n\nZůstala po ní v zemi díra, že by se tam vešel povoz i s koňmi...\n\nJak řepa povolila, svalili se všichni na jednu hromadu, smáli se a smáli. Společnými silami se jim dílo podařilo."
docs.append({"id":"velika_repa","name":"O veliké řepě – rozšířená verze","group":"reference","source":"uživatelsky dodaný text; přesné znění vložené do notebooku","text":TURNIP_TEXT})
assert len(docs)==11
print("Celkem jedenáct celých textů: devět autorských a dvě reference.")

## 4 · Kontrola shody s Pilot v2

Ověříme ID devíti karet a rozměry uložených vektorů. Celotextové vektory zůstávají původní.

In [ ]:
baseline_by_id={x.get("card_id",x.get("id")):x for x in baseline_vectors}
assert set(baseline_by_id)==set(CARDS), "ID karet v ZIPu neodpovídají očekávaným kartám."
for d in docs:
 if d["group"]=="authored":
  b=baseline_by_id[d["id"]]
  print(d["name"],"| ZIP:",len(b.get("text","")),"znaků | nyní:",len(d["text"]),"znaků")
print("Kontrola ID a rozměrů dokončena.")

## 5 · Strukturální a lexikální profily celých příběhů

Slovní formy sjednocujeme malými písmeny; různé pády zůstávají oddělené. Počítáme tokeny, unikátní formy, TTR, hapax, opakované typy, podíl opakovaných tokenů, frekvenční entropii, věty, odstavce a zakončení vět.

In [ ]:
WORD_RE=re.compile(r"[^\W\d_]+(?:[’'-][^\W\d_]+)*",re.UNICODE)
SENT_RE=re.compile(r"[^.!?…]+(?:[.!?…]+|$)",re.S)
def sentence_spans(text):
 out=[]
 for m in SENT_RE.finditer(text):
  s=m.group().strip()
  if s:
   a=m.start()+len(m.group())-len(m.group().lstrip()); out.append((a,a+len(s),s))
 return out
profiles=[]; occurrence_rows=[]
for d in docs:
 text=d["text"]; matches=list(WORD_RE.finditer(text))
 tokens=[m.group().casefold() for m in matches]; counts=Counter(tokens)
 total=len(tokens); unique=len(counts); hapax=sum(v==1 for v in counts.values())
 probs=np.array(list(counts.values()),dtype=float)/max(1,total)
 entropy=float(-(probs*np.log2(probs)).sum()) if total else 0.0
 spans=sentence_spans(text); paragraphs=[p for p in re.split(r"\n\s*\n",text) if p.strip()]
 repeated=defaultdict(list)
 for m in matches:
  form=m.group().casefold()
  if counts[form]>1:
   si=next((i for i,(a,b,s) in enumerate(spans) if a<=m.start()<b),None)
   if si is not None:
    a,b,s=spans[si]
    repeated[form].append({"sentence":s,"sentence_index":si,"relative_position":m.start()/max(1,len(text)),"character_position":m.start()})
 repeated_forms=[{"form":f,"count":len(rows),"occurrences":rows} for f,rows in repeated.items()]
 p={"id":d["id"],"name":d["name"],"group":d["group"],"source":d["source"],"characters":len(text),"word_tokens":total,"unique_forms":unique,"TTR":unique/total if total else 0,"hapax_count":hapax,"hapax_ratio":hapax/unique if unique else 0,"repeated_types":sum(v>1 for v in counts.values()),"repeated_token_share":sum(v for v in counts.values() if v>1)/total if total else 0,"frequency_entropy_bits":entropy,"sentence_count":len(spans),"paragraph_count":len(paragraphs),"sentence_endings":len(re.findall(r"[.!?…]+",text)),"top_words":counts.most_common(50),"repeated_forms":repeated_forms}
 profiles.append(p)
 for item in repeated_forms:
  for occ in item["occurrences"]: occurrence_rows.append({"doc_id":d["id"],"doc_name":d["name"],"group":d["group"],"form":item["form"],**occ})
profile_cols=["name","group","characters","word_tokens","unique_forms","TTR","hapax_count","hapax_ratio","repeated_types","repeated_token_share","frequency_entropy_bits","sentence_count","paragraph_count","sentence_endings"]
profile_df=pd.DataFrame([{k:p[k] for k in profile_cols} for p in profiles])
display(profile_df)
print("Výskyty opakovaných forem:",len(occurrence_rows))

## 6 · Větné kontexty opakovaných slov

Každý výskyt opakované formy je propojen s celou větou, indexem věty a relativní polohou v příběhu. Věta může obsahovat více opakovaných forem.

In [ ]:
occurrence_df=pd.DataFrame(occurrence_rows)
display(occurrence_df.head(30))
print("Unikátní dvojice příběh–věta:",len({(r["doc_id"],r["sentence"]) for r in occurrence_rows}))
print("Unikátní opakované formy:",len({(r["doc_id"],r["form"]) for r in occurrence_rows}))

## 7 · BGE-M3: nové větné embeddingy

Každá unikátní věta s opakovanou formou dostane jeden embedding. Věta se v rámci příběhu zakóduje jen jednou. Zpracování probíhá po menších dávkách s průběžným výpisem. Toto je nově počítaná část experimentu.

In [ ]:
MODEL_NAME="BAAI/bge-m3"
print("Načítám model:",MODEL_NAME)
model=BGEM3FlagModel(MODEL_NAME,use_fp16=bool(torch.cuda.is_available()))
print("Model načten.")
unique_sentences={}
for row in occurrence_rows:
 key=(row["doc_id"],row["sentence"])
 if key not in unique_sentences: unique_sentences[key]=len(unique_sentences)
sentence_texts=[None]*len(unique_sentences)
for (doc_id,sentence),idx in unique_sentences.items(): sentence_texts[idx]=sentence
print("Unikátní větné kontexty:",len(sentence_texts))
sentence_vectors=[]
BATCH_SIZE=8 if torch.cuda.is_available() else 4
for start in range(0,len(sentence_texts),BATCH_SIZE):
 batch=sentence_texts[start:start+BATCH_SIZE]
 enc=model.encode(batch,batch_size=len(batch),max_length=512,return_dense=True,return_sparse=False,return_colbert_vecs=False)
 sentence_vectors.extend(np.asarray(enc["dense_vecs"],dtype=np.float32))
 print(f"Zakódováno {min(start+len(batch),len(sentence_texts))}/{len(sentence_texts)} vět")
sentence_vectors=np.asarray(sentence_vectors,dtype=np.float32)
assert sentence_vectors.shape==(len(sentence_texts),1024),f"Chybný tvar: {sentence_vectors.shape}"
for row in occurrence_rows: row["vector_index"]=unique_sentences[(row["doc_id"],row["sentence"])]
print("Embeddingy hotové:",sentence_vectors.shape)

## 8 · Párové podobnosti a kontextová proměnlivost

U každé opakované formy porovnáváme všechny dvojice odlišných větných kontextů v rámci jednoho příběhu. Uchováváme jednotlivé páry, průměr, směrodatnou odchylku i odlišnost. Kontextová odlišnost není přímým měřením lexikální polysémie.

In [ ]:
def cosine_similarity(a,b):
 den=float(np.linalg.norm(a)*np.linalg.norm(b))
 return float(np.dot(a,b)/den) if den else float("nan")
groups=defaultdict(list)
for row in occurrence_rows: groups[(row["doc_id"],row["form"])].append(row)
form_results=[]; pair_results=[]
for (doc_id,form),rows in groups.items():
 distinct=[]; seen=set()
 for row in rows:
  if row["sentence"] not in seen: seen.add(row["sentence"]); distinct.append(row)
 if len(distinct)<2: continue
 sims=[]
 for i in range(len(distinct)):
  for j in range(i+1,len(distinct)):
   sim=cosine_similarity(sentence_vectors[distinct[i]["vector_index"]],sentence_vectors[distinct[j]["vector_index"]])
   sims.append(sim)
   pair_results.append({"doc_id":doc_id,"form":form,"sentence_a":distinct[i]["sentence"],"sentence_b":distinct[j]["sentence"],"relative_position_a":distinct[i]["relative_position"],"relative_position_b":distinct[j]["relative_position"],"cosine_similarity":sim,"cosine_dissimilarity":1-sim})
 p=next(x for x in profiles if x["id"]==doc_id)
 form_results.append({"doc_id":doc_id,"doc_name":p["name"],"group":p["group"],"form":form,"occurrences":len(rows),"distinct_sentence_contexts":len(distinct),"pair_count":len(sims),"mean_context_similarity":float(np.mean(sims)),"std_context_similarity":float(np.std(sims)),"mean_context_dissimilarity":float(1-np.mean(sims)),"contexts":[{"sentence":r["sentence"],"relative_position":r["relative_position"],"sentence_index":r["sentence_index"]} for r in distinct]})
form_df=pd.DataFrame(form_results); pair_df=pd.DataFrame(pair_results)
print("Forem s více kontexty:",len(form_df),"| dvojic:",len(pair_df))
if len(form_df): display(form_df.sort_values("mean_context_dissimilarity",ascending=False).head(40))

## 9 · Souhrny po příbězích a skupinách

In [ ]:
summary_rows=[]
for p in profiles:
 f=form_df[form_df.doc_id==p["id"]] if len(form_df) else pd.DataFrame()
 summary_rows.append({"id":p["id"],"name":p["name"],"group":p["group"],"tokens":p["word_tokens"],"unique_forms":p["unique_forms"],"TTR":p["TTR"],"hapax_ratio":p["hapax_ratio"],"repeated_types":p["repeated_types"],"repeated_token_share":p["repeated_token_share"],"forms_with_multiple_contexts":len(f),"mean_context_similarity":float(f.mean_context_similarity.mean()) if len(f) else np.nan,"mean_context_dissimilarity":float(f.mean_context_dissimilarity.mean()) if len(f) else np.nan})
summary_df=pd.DataFrame(summary_rows)
display(summary_df)
display(summary_df.groupby("group").mean(numeric_only=True))

## 10 · Celotextové vektory a lidová reference

Původní vektory devíti autorských příběhů a kohoutka jsou převzaty z Pilot v2. Pro novou referenci O veliké řepě počítáme jeden nový celotextový embedding stejným modelem BGE-M3. Obě reference pak porovnáváme s devíti autorskými příběhy. Výsledky původního Pilot v2 zůstávají zachovány.

In [ ]:
reference_vector=np.asarray(baseline_reference["vector"],dtype=np.float32)
# Celotextový embedding nové reference se počítá stejným modelem a nastavením.
turnip_doc=next(d for d in docs if d["id"]=="velika_repa")
turnip_encoding=model.encode([turnip_doc["text"]],batch_size=1,max_length=8192,return_dense=True,return_sparse=False,return_colbert_vecs=False)
turnip_vector=np.asarray(turnip_encoding["dense_vecs"][0],dtype=np.float32)
assert turnip_vector.shape==(1024,)
reference_vectors={"kohoutek_slepicka":reference_vector,"velika_repa":turnip_vector}
whole_story_comparisons=[]
for item in baseline_vectors:
 vector=np.asarray(item["vector"],dtype=np.float32)
 for ref_id,ref_vector in reference_vectors.items():
  whole_story_comparisons.append({"id":item.get("card_id",item.get("id")),"name":item.get("card_name",item.get("card_id",item.get("id"))),"reference_id":ref_id,"cosine_similarity":cosine_similarity(vector,ref_vector),"cosine_distance":1-cosine_similarity(vector,ref_vector)})
for ref_id,ref_vector in reference_vectors.items():
 whole_story_comparisons.append({"id":ref_id,"name":ref_id,"reference_id":ref_id,"cosine_similarity":1.0,"cosine_distance":0.0})
whole_story_df=pd.DataFrame(whole_story_comparisons).sort_values("cosine_similarity_to_folk_reference",ascending=False)
display(whole_story_df)

## 11 · Vizualizace návratů v celém příběhu

Každý řádek je opakovaná slovní forma, body ukazují polohu jejích výskytů od začátku do konce příběhu. Nejde o významové štítky.

In [ ]:
for p in profiles:
 rows=[(x["form"],o["relative_position"]) for x in p["repeated_forms"] for o in x["occurrences"]]
 if not rows: continue
 top=[x["form"] for x in sorted(p["repeated_forms"],key=lambda x:x["count"],reverse=True)[:22]]
 fig,ax=plt.subplots(figsize=(12,max(4,len(top)*.3)))
 for yi,form in enumerate(top):
  xs=[pos for f,pos in rows if f==form]; ax.scatter(xs,[yi]*len(xs),s=24)
 ax.set_yticks(range(len(top)),top); ax.set_xlim(0,1)
 ax.set_xlabel("Poloha v celém textu (0 = začátek, 1 = konec)")
 ax.set_title("Návraty opakovaných slov · "+p["name"]); ax.grid(axis="x",alpha=.25)
 plt.tight_layout(); plt.show()

## 12 · Vizualizace kontextové proměnlivosti

In [ ]:
if len(form_df):
 plot_df=form_df.groupby(["doc_name","group"],as_index=False).agg(mean_dissimilarity=("mean_context_dissimilarity","mean"),forms=("form","count"),pairs=("pair_count","sum")).sort_values("mean_dissimilarity")
 display(plot_df)
 fig,ax=plt.subplots(figsize=(11,5))
 ax.bar(plot_df["doc_name"],plot_df["mean_dissimilarity"])
 ax.set_ylabel("Průměrná kontextová odlišnost (1 − cosine)")
 ax.set_title("Kontextová proměnlivost opakovaných forem")
 ax.tick_params(axis="x",rotation=35); ax.grid(axis="y",alpha=.2)
 plt.tight_layout(); plt.show()
else: print("Není dost forem s více větnými kontexty.")

## 13 · Dvanáct úhlů Semantic Gap

Úhly nejsou automaticky přiřazovány BGE-M3 a nejsou embeddingovými dimenzemi. Jedno místo může být relevantní z více úhlů. Jde o interpretační rámec pro následnou anotaci.

In [ ]:
ANGLES=["Analytický","Kulturní","Filozofický","Naivní","Jazykový","Metaforický","Asociační","Introspektivní","Ironický","Groteskní","Absurdní","Surrealistický"]
angle_framework={"principle":"Interpretační perspektivy, nikoli automatické třídy slov ani měřené dimenze.","multiple_angles_per_occurrence_allowed":True,"annotation_status":"not_annotated","angles":[{"id":i+1,"name":a} for i,a in enumerate(ANGLES)]}
print(json.dumps(angle_framework,ensure_ascii=False,indent=2))

## 14 · Export kompletních výsledků

Exportujeme profily, výskyty, větné vektory, párová srovnání, skupinové souhrny a celotextové porovnání. Výstupy mají vlastní označení Experiment 2 v3. Pilot v2 zůstává nedotčen.

In [ ]:
profiles_payload={"experiment":"MOUSE CORE Experiment 2 v3","profiles":profiles,"angle_framework":angle_framework,"method_notes":["Celý příběh zůstává základní jednotkou analýzy.","Věty slouží jako lokální kontexty opakovaných slovních forem.","TTR měří lexikální rozmanitost forem, nikoli významovou hloubku.","Větné embeddingy reprezentují celé věty, nikoli izolovaná slova.","Kontextová odlišnost nedokazuje polysémii ani významový posun.","Celotextové vektory jsou převzaty z Pilot v2 a znovu se nepočítají."]}
context_vectors_payload={"model":MODEL_NAME,"description":"Vektory celých vět obsahujících opakované formy; nejde o samostatné vektory slov.","unique_sentence_vectors":[{"doc_id":did,"sentence":s,"vector":[float(x) for x in sentence_vectors[i]]} for (did,s),i in unique_sentences.items()],"occurrences":occurrence_rows}
variability_payload={"model":MODEL_NAME,"measure":"Párová kosinová podobnost a odlišnost větných embeddingů stejné opakované formy v jednom příběhu.","form_results":form_results,"pair_results":pair_results,"warning":"Kontextová proměnlivost není přímým měřením různých lexikálních významů."}
whole_payload={"source":"MOUSE_CORE_v2_vysledky.zip","vectors":baseline_vectors,"reference":baseline_reference,"comparisons":whole_story_comparisons}
(OUT/"mouse_core_experiment2_v3_profiles.json").write_text(json.dumps(profiles_payload,ensure_ascii=False,indent=2),encoding="utf-8")
(OUT/"mouse_core_experiment2_v3_context_vectors.json").write_text(json.dumps(context_vectors_payload,ensure_ascii=False),encoding="utf-8")
(OUT/"mouse_core_experiment2_v3_context_variability.json").write_text(json.dumps(variability_payload,ensure_ascii=False,indent=2),encoding="utf-8")
(OUT/"mouse_core_experiment2_v3_whole_story_comparison.json").write_text(json.dumps(whole_payload,ensure_ascii=False),encoding="utf-8")
profile_df.to_csv(OUT/"mouse_core_experiment2_v3_lexical_profiles.csv",index=False,encoding="utf-8-sig")
occurrence_df.to_csv(OUT/"mouse_core_experiment2_v3_occurrences.csv",index=False,encoding="utf-8-sig")
form_df.drop(columns=["contexts"],errors="ignore").to_csv(OUT/"mouse_core_experiment2_v3_context_variability.csv",index=False,encoding="utf-8-sig")
pair_df.to_csv(OUT/"mouse_core_experiment2_v3_context_pairs.csv",index=False,encoding="utf-8-sig")
summary_df.to_csv(OUT/"mouse_core_experiment2_v3_summary.csv",index=False,encoding="utf-8-sig")
whole_story_df.to_csv(OUT/"mouse_core_experiment2_v3_whole_story_similarity.csv",index=False,encoding="utf-8-sig")
zip_path=OUT/"MOUSE_CORE_Experiment2_v3_vysledky.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
 for path in sorted(OUT.iterdir()):
  if path.is_file() and path!=zip_path: zf.write(path,arcname=path.name)
print("Export dokončen:",zip_path)
print("Počet výstupních souborů:",len(list(OUT.iterdir())))
files.download(str(zip_path))

## 15 · Kontrolní souhrn a metodické hranice

Experiment je dokončen, pokud je načteno všech deset textů, převzato devět původních celotextových vektorů a vektor reference, vytvořeny lexikální profily a kontexty, zakódovány všechny větné kontexty, vypočteny párové podobnosti a vytvořen ZIP exportů.

Číselné rozdíly mezi autorskou a lidovou skupinou samy o sobě nepotvrzují hypotézu o sémantické hustotě. K tomu je nutná interpretace konkrétních míst v příbězích, ruční anotace úhlů Semantic Gap a ověření shody anotací.